# Financial Sentiment Analysis with FinBERT

This notebook fine-tunes [FinBERT](https://huggingface.co/ProsusAI/finbert) on the
[financial_phrasebank](https://huggingface.co/datasets/financial_phrasebank) dataset
for three-class sentiment classification: **positive**, **negative**, **neutral**.

## 1. Setup & Imports

In [ ]:
!pip install -q transformers datasets scikit-learn pandas torch tqdm

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

SEED = 42
MODEL_NAME = "ProsusAI/finbert"
NUM_LABELS = 3
LABEL2ID = {"positive": 0, "negative": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Dataset

We use the `sentences_allagree` split of **financial_phrasebank** from HuggingFace,
which contains only sentences where all annotators agreed on the sentiment label.

Dataset link: https://huggingface.co/datasets/financial_phrasebank

In [ ]:
raw = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
df = raw["train"].to_pandas()

# financial_phrasebank uses integer labels 0=negative, 1=neutral, 2=positive
# Remap to our convention: positive=0, negative=1, neutral=2
fp_label_map = {0: "negative", 1: "neutral", 2: "positive"}
df["label"] = df["label"].map(fp_label_map)
df.rename(columns={"sentence": "sentence"}, inplace=True)

print(df["label"].value_counts())
df.head()

## 3. Train / Val / Test Split

In [ ]:
train_val, test_df = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df["label"])
train_df, val_df = train_test_split(train_val, test_size=0.111, random_state=SEED, stratify=train_val["label"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

os.makedirs("../data", exist_ok=True)
train_df.to_csv("../data/train.csv", index=False)
val_df.to_csv("../data/val.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

## 4. Tokenisation & PyTorch Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FinancialDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        encoding = self.tokenizer(
            row["sentence"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(LABEL2ID[row["label"]], dtype=torch.long),
        }

train_dataset = FinancialDataset(train_df, tokenizer)
val_dataset   = FinancialDataset(val_df,   tokenizer)
test_dataset  = FinancialDataset(test_df,  tokenizer)

## 5. Fine-tune FinBERT with HuggingFace Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

training_args = TrainingArguments(
    output_dir="../results/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="../results/logs",
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    seed=SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    report = classification_report(
        labels, predictions, target_names=list(LABEL2ID.keys()), output_dict=True, zero_division=0
    )
    return {
        "accuracy": report["accuracy"],
        "f1_macro": report["macro avg"]["f1-score"],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluate on Test Set

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

report = classification_report(
    true_labels, preds,
    target_names=list(LABEL2ID.keys()),
    output_dict=True,
    zero_division=0,
)
print(classification_report(true_labels, preds, target_names=list(LABEL2ID.keys()), zero_division=0))

## 7. Save Results Table

In [ ]:
results_df = pd.DataFrame([{
    "model": "FinBERT (fine-tuned)",
    "accuracy": round(report["accuracy"], 4),
    "f1_macro": round(report["macro avg"]["f1-score"], 4),
    "f1_positive": round(report["positive"]["f1-score"], 4),
    "f1_negative": round(report["negative"]["f1-score"], 4),
    "f1_neutral":  round(report["neutral"]["f1-score"], 4),
}])

os.makedirs("../results", exist_ok=True)
results_df.to_csv("../results/results_table.csv", index=False)
print(results_df)